In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
DATA_PATH = Path("../data/raw/PS_20174392719_1491204439457_log.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
base_features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud",
    "isFraud"
]

feature_df = df[base_features].copy()

#### Origin balance change

##### The sender's balance changes from:

In [15]:
feature_df["balance_change_orig"] =(
    feature_df["oldbalanceOrg"] - feature_df["newbalanceOrig"]
) 

In [14]:
feature_df["balance_change_dest"] = (
    feature_df["newbalanceDest"]
    - feature_df["oldbalanceDest"]
)

##### Origin balance error

##### A useful fraud signal is whether:
old sender balance
       -
new sender balance

In [13]:
feature_df["balance_error_orig"] = (
    feature_df["oldbalanceOrg"]
    - feature_df["amount"]
    - feature_df["newbalanceOrig"]
)

##### Destination balance error

In [6]:
feature_df["balance_error_dest"] = (
    feature_df["oldbalanceDest"]
    + feature_df["amount"]
    - feature_df["newbalanceDest"]
)

##### Amount-to-origin-balance ratio

In [7]:
feature_df["amount_to_old_balance_ratio"] = (
    feature_df["amount"]
    / (feature_df["oldbalanceOrg"] + 1)
)

##### Amount-to-new-balance ratio

In [8]:
feature_df["amount_to_new_balance_ratio"] = (
    feature_df["amount"]
    / (feature_df["newbalanceOrig"] + 1)
)

##### Destination amount ratio

In [9]:
feature_df["amount_to_old_dest_balance_ratio"] = (
    feature_df["amount"]
    / (feature_df["oldbalanceDest"] + 1)
)

##### Time features

PaySim's step represents simulation time.

a rough hour-of-day representation:

In [10]:
feature_df["transaction_hour"] = (
    feature_df["step"] % 24
)

feature_df["transaction_day"] = (
    feature_df["step"] // 24
)

In [16]:
new_features = [
    "balance_change_orig",
    "balance_change_dest",
    "balance_error_orig",
    "balance_error_dest",
    "amount_to_old_balance_ratio",
    "amount_to_new_balance_ratio",
    "amount_to_old_dest_balance_ratio",
    "transaction_hour",
    "transaction_day"
]

feature_df[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
balance_change_orig,6362620.0,-21230.564504,146643.286487,-1915267.90,0.000000,0.000000,10150.435,1.000000e+07
balance_change_dest,6362620.0,124294.731682,812939.081073,-13060826.21,0.000000,0.000000,149105.425,1.056878e+08
balance_error_orig,6362620.0,-201092.468053,606650.460140,-92445516.64,-249641.087500,-68677.255000,-2954.230,1.000000e-02
balance_error_dest,6362620.0,55567.171867,441528.768827,-75885725.63,0.000000,3500.490000,29353.045,1.319123e+07
amount_to_old_balance_ratio,6362620.0,70674.476792,508424.269379,0.00,0.234401,6.453832,12287.765,9.244552e+07
amount_to_new_balance_ratio,6362620.0,136613.064270,604439.687607,0.00,0.261557,7557.495000,134263.970,9.244552e+07
amount_to_old_dest_balance_ratio,6362620.0,27883.832244,187930.639208,0.00,0.160704,0.917831,9732.300,6.096528e+07
transaction_hour,6362620.0,15.321454,4.321799,0.00,12.000000,16.000000,19.000,2.300000e+01
transaction_day,6362620.0,9.503158,5.922111,0.00,6.000000,9.000000,13.000,3.000000e+01


##### Check for infinite values

Ratios can sometimes produce infinite values if the denominator is zero.

In [17]:
np.isinf(
    feature_df[new_features]
).sum()

balance_change_orig                 0
balance_change_dest                 0
balance_error_orig                  0
balance_error_dest                  0
amount_to_old_balance_ratio         0
amount_to_new_balance_ratio         0
amount_to_old_dest_balance_ratio    0
transaction_hour                    0
transaction_day                     0
dtype: int64

In [18]:
feature_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

In [19]:
print(
    feature_df[new_features]
    .isnull()
    .sum()
)

balance_change_orig                 0
balance_change_dest                 0
balance_error_orig                  0
balance_error_dest                  0
amount_to_old_balance_ratio         0
amount_to_new_balance_ratio         0
amount_to_old_dest_balance_ratio    0
transaction_hour                    0
transaction_day                     0
dtype: int64


In [20]:
feature_comparison = (
    feature_df
    .groupby("isFraud")[new_features]
    .median()
)

feature_comparison.T

isFraud,0,1
balance_change_orig,0.000000,436317.490000
balance_change_dest,0.000000,0.000000
balance_error_orig,-69049.310000,0.000000
balance_error_dest,3500.680000,2231.460000
amount_to_old_balance_ratio,6.511566,0.999998
amount_to_new_balance_ratio,7487.060000,401529.350000
amount_to_old_dest_balance_ratio,0.915134,116419.580000
transaction_hour,16.000000,12.000000
transaction_day,9.000000,15.000000


In [21]:
correlation_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    *new_features
]

correlation = feature_df[
    correlation_features + ["isFraud"]
].corr()["isFraud"].sort_values(
    ascending=False
)

print(correlation)

isFraud                             1.000000
balance_change_orig                 0.362472
amount_to_old_dest_balance_ratio    0.180205
amount                              0.076688
amount_to_new_balance_ratio         0.068333
balance_error_dest                  0.055120
transaction_day                     0.032577
balance_change_dest                 0.027028
balance_error_orig                  0.011283
oldbalanceOrg                       0.010154
newbalanceDest                      0.000535
amount_to_old_balance_ratio        -0.004915
oldbalanceDest                     -0.005885
newbalanceOrig                     -0.008148
transaction_hour                   -0.031403
Name: isFraud, dtype: float64


In [22]:
print("Original columns:", len(base_features))
print("Engineered columns:", len(new_features))
print("Total columns:", feature_df.shape[1])

feature_df.head()

Original columns: 9
Engineered columns: 9
Total columns: 19


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,isFraud,balance_change_Orig,balance_error_orig,balance_error_dest,amount_to_old_balance_ratio,amount_to_new_balance_ratio,amount_to_old_dest_balance_ratio,transaction_hour,transaction_day,balance_change_dest,balance_change_orig
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0,9839.64,0.0,9839.64,0.057834,0.061384,9839.640000,1,0,0.0,9839.64
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0,1864.28,0.0,1864.28,0.087731,0.096168,1864.280000,1,0,0.0,1864.28
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,0,1,181.00,0.0,181.00,0.994505,181.000000,181.000000,1,0,0.0,181.00
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,0,1,181.00,0.0,21363.00,0.994505,181.000000,0.008545,1,0,-21182.0,181.00
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0,11668.14,0.0,11668.14,0.280788,0.390410,11668.140000,1,0,0.0,11668.14
